# Notebook 00 — Setup & EDA
Verify environment, load DialogSum, explore data statistics.

## 1. Environment Check

In [ ]:
import subprocess, sys

# Install PyTorch (CUDA 12.4) then remaining packages
# Run once; comment out after first execution
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1',
#                 '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt'], check=True)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print('BF16 supported:', torch.cuda.is_bf16_supported())


## 2. Load Dataset

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.loader import load_dialogsum

dataset = load_dialogsum(cache_dir='../data/raw/dialogsum')
print(dataset)


In [ ]:
# Show a random sample
import random
sample = random.choice(dataset['train'])
print('--- DIALOGUE ---')
print(sample['dialogue'])
print('\n--- SUMMARY ---')
print(sample['summary'])
print('\nTopic:', sample.get('topic', 'N/A'))


## 3. Data Statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def word_count(text): return len(text.split())

train_df = pd.DataFrame(dataset['train'])
train_df['dialogue_words'] = train_df['dialogue'].apply(word_count)
train_df['summary_words']  = train_df['summary'].apply(word_count)
train_df['compression']    = train_df['dialogue_words'] / train_df['summary_words']

print(train_df[['dialogue_words', 'summary_words', 'compression']].describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(train_df['dialogue_words'], bins=40, color='#4C72B0', edgecolor='white')
axes[0].set_title('Dialogue Length (words)')
axes[0].set_xlabel('Words'); axes[0].set_ylabel('Count')

axes[1].hist(train_df['summary_words'], bins=30, color='#DD8452', edgecolor='white')
axes[1].set_title('Summary Length (words)')
axes[1].set_xlabel('Words')

axes[2].hist(train_df['compression'], bins=30, color='#55A868', edgecolor='white')
axes[2].set_title('Compression Ratio (dialogue/summary)')
axes[2].set_xlabel('Ratio')

plt.tight_layout()
plt.savefig('../results/figures/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Speaker tag frequency
import re
all_dialogues = ' '.join(dataset['train']['dialogue'])
speakers = re.findall(r'#Person(\d+)#', all_dialogues)
from collections import Counter
print('Speaker distribution:', Counter(speakers).most_common(5))


In [ ]:
# Fraction of samples exceeding PEGASUS 512-token limit (rough word estimate)
over_512 = (train_df['dialogue_words'] > 400).sum()  # 400 words ≈ 512 tokens
print(f'Samples likely truncated by PEGASUS: {over_512}/{len(train_df)} '
      f'({100*over_512/len(train_df):.1f}%)')
